In [15]:
import re
import random
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

In [16]:
# Load Data
PROC = "../Data/processed" 

# Load real notes from test set
print("Loading real notes...")
with open(f"{PROC}/test.txt", "r", encoding="utf-8") as f:
    real_texts = f.read().split("\n<|endoftext|>\n")
real_texts = [t.strip() for t in real_texts if t.strip()]
print(f"Loaded {len(real_texts)} real notes")

# Load synthetic notes
qwen_df = pd.read_csv(f"{PROC}/synthetic_notes_qwen_rewrite.csv")
llama_df = pd.read_csv(f"{PROC}/synthetic_notes_llama_rewrite.csv")

print(f"Qwen: {len(qwen_df)}, LLaMA: {len(llama_df)}")

Loading real notes...
Loaded 33991 real notes
Qwen: 1000, LLaMA: 1000


In [17]:
def strip_structure(text):
    """Remove de-identification and MIMIC header scaffolding so the classifier
    is forced onto clinical prose rather than formatting artifacts."""
    text = str(text)
    text = re.sub(r'<?PHI\w*', ' ', text)
    text = re.sub(r'\*\*', '', text)          # remove markdown bold
    text = re.sub(r'#{1,6}\s*', '', text)

    header_pattern = (
        r'(?im)^\s*(Name|Unit No|Unit Number|Admission Date|Discharge Date|Date of Birth|Sex|'
        r'Gender|Service|Attending|Attending Physician|Allergies|Allergy|Followup Instructions|'
        r'Discharge Disposition):.*$'
    )
    text = re.sub(header_pattern, ' ', text)

    clinical = (r'(?i)(Chief Complaint|History of Present Illness|Past Medical History|'
                r'Physical Exam|Social History|Family History|Pertinent Results|'
                r'Brief Hospital Course|Discharge Medications|Discharge Instructions|'
                r'Discharge Diagnosis|Discharge Condition|Medications on Admission|'
                r'Major Surgical or Invasive Procedure)\s*:?')
    
    text = re.sub(clinical, ' ', text)
    # Bare header-label words that survive as fragments (sex service, unit no, birth sex...)

    labels = (r'(?i)\b(Sex|Service|Unit No|Unit Number|Admission Date|Discharge Date|'
              r'Date of Birth|Birth|Attending|Disposition|Followup|Admission)\b')
    text = re.sub(labels, ' ', text)
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', ' ', text)
    text = re.sub(r'\s+', ' ', text)   # tidy the whitespace the removals leave behind
    return text

CONNECTORS = ["however","furthermore","additionally","moreover","therefore",
              "consequently","subsequently","notably","overall","importantly","specifically"]

def extract_features(text):
    text=str(text); sents=[s for s in re.split(r'[.!?]',text) if s.strip()]
    sl=[len(s.split()) for s in sents]; words=text.split(); nw=len(words) or 1; nc=len(text) or 1; tl=text.lower()
    f={}
    f["word_count"]=len(words); f["sentence_count"]=len(sents)
    f["avg_sentence_length"]=np.mean(sl) if sl else 0
    f["sentence_length_std"]=np.std(sl) if len(sl)>1 else 0
    f["sentence_length_cv"]=f["sentence_length_std"]/f["avg_sentence_length"] if f["avg_sentence_length"]>0 else 0
    f["vocab_richness"]=len(set(w.lower() for w in words))/nw
    f["uppercase_word_ratio"]=sum(1 for w in words if w.isupper())/nw
    f["comma_density"]=text.count(",")/nc*1000; f["colon_density"]=text.count(":")/nc*1000
    f["semicolon_density"]=text.count(";")/nc*1000; f["paren_density"]=text.count("(")/nc*1000
    syll=sum(max(1,len(re.findall(r'[aeiouyAEIOUY]+',w))) for w in words)
    f["flesch"]=206.835-1.015*(nw/max(1,len(sents)))-84.6*(syll/nw)
    f["connector_density"]=sum(tl.count(c) for c in CONNECTORS)/nw*100
    return f

In [18]:
# Building a balanced dataset
def build_dataset(real_texts, synthetic_df, n_real=1000, seed=42):
    random.seed(seed)
    sampled_real = random.sample(real_texts, n_real)
    texts = sampled_real + synthetic_df["generated_text"].astype(str).tolist()
    labels = [0] * n_real + [1] * len(synthetic_df)
    return texts, labels

qwen_texts, qwen_labels = build_dataset(real_texts, qwen_df)
llama_texts, llama_labels = build_dataset(real_texts, llama_df)

In [19]:
# Split datasets and fit TF-IDF on train test
def prepare_data(texts, labels):
    # Split on raw texts first to avoid TF-IDF leakage from test into train
    X_train_texts, X_test_texts, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=labels)

    X_train_texts = [strip_structure(t) for t in X_train_texts]
    X_test_texts  = [strip_structure(t) for t in X_test_texts]

    tfidf = TfidfVectorizer(max_features=500, min_df=5, ngram_range=(1, 2), sublinear_tf=True)
    tfidf_train = tfidf.fit_transform(X_train_texts)
    tfidf_test = tfidf.transform(X_test_texts)

    hand_train = pd.DataFrame([extract_features(t) for t in X_train_texts])
    hand_test = pd.DataFrame([extract_features(t) for t in X_test_texts])
    feature_names = tfidf.get_feature_names_out().tolist() + hand_train.columns.tolist()

    X_train = hstack([tfidf_train, sp.csr_matrix(hand_train.values)])
    X_test = hstack([tfidf_test, sp.csr_matrix(hand_test.values)])

    return X_train, X_test, y_train, y_test, feature_names, tfidf

data_qwen = prepare_data(qwen_texts, qwen_labels)
data_llama = prepare_data(llama_texts, llama_labels)
print("Feature matrices ready")

Feature matrices ready


In [20]:
def cross_validate_generator(texts, labels, name, n_splits=5):
    texts = np.array(texts, dtype=object)
    labels = np.array(labels)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    aucs, accs = [], []
    for fold, (tr, te) in enumerate(skf.split(texts, labels)):
        tr_texts, te_texts = texts[tr], texts[te]
        tr_y, te_y = labels[tr], labels[te]

        # strip structure BEFORE tf-idf; fit tf-idf on train fold only
        tr_clean = [strip_structure(t) for t in tr_texts]
        te_clean = [strip_structure(t) for t in te_texts]

        tfidf = TfidfVectorizer(max_features=500, min_df=5,
                                ngram_range=(1, 2), sublinear_tf=True)
        Xtr_tfidf = tfidf.fit_transform(tr_clean)
        Xte_tfidf = tfidf.transform(te_clean)

        # hand-crafted features on ORIGINAL text (structure matters as signal here,
        # but PHI/date features already removed inside extract_features)
        hand_tr = pd.DataFrame([extract_features(t) for t in tr_texts])
        hand_te = pd.DataFrame([extract_features(t) for t in te_texts])

        Xtr = hstack([Xtr_tfidf, csr_matrix(hand_tr.values)])
        Xte = hstack([Xte_tfidf, csr_matrix(hand_te.values)])

        clf = XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.3,
            reg_alpha=1.0, reg_lambda=2.0,
            eval_metric="logloss", random_state=42
        )
        clf.fit(Xtr, tr_y)
        prob = clf.predict_proba(Xte)[:, 1]
        pred = (prob >= 0.5).astype(int)
        aucs.append(roc_auc_score(te_y, prob))
        accs.append(accuracy_score(te_y, pred))

    print(f"{name:12s}  AUC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}   "
          f"Acc {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    return {"generator": name, "auc_mean": np.mean(aucs), "auc_std": np.std(aucs),
            "acc_mean": np.mean(accs), "acc_std": np.std(accs)}

results = []
for name, df in [("Qwen", qwen_df), ("LLaMA", llama_df)]:
    texts, labels = build_dataset(real_texts, df)   # your existing builder
    results.append(cross_validate_generator(texts, labels, name))

pd.DataFrame(results)

Qwen          AUC 1.000 ± 0.000   Acc 0.999 ± 0.001
LLaMA         AUC 1.000 ± 0.000   Acc 1.000 ± 0.001


,generator,auc_mean,auc_std,acc_mean,acc_std
0,Qwen,1.0,4.965068e-17,0.9990,0.001225
1,LLaMA,1.0,4.965068e-17,0.9995,0.001000


In [21]:
def fit_and_bundle(data_tuple, texts, labels, name):
    X_train, X_test, y_train, y_test, feature_names, tfidf = data_tuple

    clf = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.3,
        reg_alpha=1.0, reg_lambda=2.0,
        eval_metric="logloss", random_state=42
    )
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print(f"{name}: AUC {roc_auc_score(y_test, prob):.4f}  Acc {accuracy_score(y_test, pred):.4f}")

    # rebuild the RAW test texts + hand-crafted feature frames the bundle needs
    # (re-run the same split with the same seed to recover raw text + hand features)
    Xtr_texts, Xte_texts, ytr, yte = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=labels)
    hand_names = [k for k in extract_features("x").keys()]
    hte = pd.DataFrame([extract_features(t) for t in Xte_texts])   # on raw text
    htr = pd.DataFrame([extract_features(t) for t in Xtr_texts])

    joblib.dump({
        "clf": clf, "tfidf": tfidf,
        "feat_names": feature_names, "hand_names": hand_names,
        "Xte": X_test, "te_x": Xte_texts, "te_y": list(y_test),
        "hte": hte, "htr": htr, "tr_y": list(ytr),
        "prob": prob, "pred": pred,
    }, f"xgb_bundle_rewrite_{name}.joblib")
    print(f"Saved xgb_bundle_rewrite_{name}.joblib")

fit_and_bundle(data_qwen,  qwen_texts,  qwen_labels,  "qwen")
fit_and_bundle(data_llama, llama_texts, llama_labels, "llama")

qwen: AUC 1.0000  Acc 0.9975
Saved xgb_bundle_rewrite_qwen.joblib
llama: AUC 1.0000  Acc 1.0000
Saved xgb_bundle_rewrite_llama.joblib
